# Demo for reducing the Euclid flagship simulations:

The full simulations are available at https://cosmohub.pic.es/catalogs/353. If you use the catalog for any publication, please use the acknowledgements listed on this website.

Notice that the catalog inherently has a cut for the Euclid h band at ${\tt euclid\_nisp\_h} < 26.6$. The full catalog covers 1 octant of the sky but the example file only contain objects on 1 pixel (out of a total of 256), and subsampled to be 1% of the original size. Notice also there is a small region (150 < RA < 155 deg., 5 < DEC < 10 deg.) where there is no magnitude or flux cut, which has significantly higher number density. This small region *is* part of the example file.

In [ ]:
# import relevant packages
import os
from rail.projects import library
from rail.projects import RailProject
import pandas as pd

check_dir = os.path.basename(os.path.abspath(os.curdir))
if check_dir == 'examples':
    os.chdir('..')

We load the project configuration for flagship below. This config file is very similar to the RomanRubin example, but you notice a few difference. 
Firstly, I changed the `CommonPaths` to the path to the flagship example catalog, which currently sits on my scratch. The subsequent file structure is similar to the RomanRubin case, so we can use the same library file, `ci_project_libraray.yaml`. I changed the `catalog_tag` under `Baseline` to `flagship` as well. Finally, I changed `IterationVars` to match the iteration pattern, in this case I only have one pixel, `1`. 

In [ ]:
# load the project configuration:
project_flagship = RailProject.load_config("examples/ci_project-flagship.yaml")

We can check if we've found the catalog we specified:

In [ ]:
catalog_files_truth = project_flagship.get_catalog_files("truth")
print(catalog_files_truth)

Yes, this is the correct directory. Now we can reduce the raw catalog to the correct format we want.
To do this, we simply changed the `reducer_class_name` to `flagship`. This is defined in the library file, and calls the `FlagshipReducer`. This reducer does the following things:
- Convert all fluxes (here, we use columns `[fluxtype]_el_model3_ext` which includes fluxes from both the continuum and the emission lines from the galaxy, and `[fluxtype]` denotes LSST $ugrizy$ and Euclid nisp y,j,h and Euclid vis) into magnitudes, and rename the columns to standard values.
- Compute the semi-major and minor axes of the galaxy in arcsec and save in the `major` and `minor` columns. Some intermediate quantites such as orientation angles are also saved.
- Apply a `gold` magnitude cut, also defined in the library, which corresponds to $i<25.5$. 

In [ ]:
project_flagship.reduce_data(
    catalog_template="truth",
    output_catalog_template="reduced",
    reducer_class_name="flagship",
    input_selection="",
    selection="gold",
)

You can check the input and output catalogs, to see if the reducer has worked:

In [ ]:
f_truth=pd.read_parquet("/pscratch/sd/q/qhang//Flagship/ci_test_flagship_v1/1/part-0.pq")
f_reduce=pd.read_parquet("/pscratch/sd/q/qhang//Flagship/ci_test_flagship_v1_gold/1/part-0.pq")

In [ ]:
f_truth

In [ ]:
f_reduce

From here on you can follow examples in `rail_project_example.ipynb` to run the rest of the pipeline.